In [8]:
import pandas as pd
import numpy as np
import re
import string
from collections import Counter
from datetime import datetime
import nltk
import textstat
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

tqdm.pandas()


nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('vader_lexicon', quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk import pos_tag
STOPWORDS = set(stopwords.words('english'))
import torch
from sentence_transformers import SentenceTransformer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Using device: cpu


In [9]:
import os
DATA_DIR = "../data"

df_ai_posts = pd.read_csv(f"{DATA_DIR}/moltbook_post.csv", low_memory=False)
df_human_posts = pd.read_csv(f"{DATA_DIR}/reddit_data2.csv", low_memory=False)

# BRIGHTDATA_PATH = f"{DATA_DIR}/brightdata_reddit.csv"
# if os.path.exists(BRIGHTDATA_PATH):
#     df_brightdata = pd.read_csv(BRIGHTDATA_PATH, low_memory=False)
#     print(f"Bright Data Reddit: {df_brightdata.shape}")
# else:
#     df_brightdata = None

print(f"AI Posts: {df_ai_posts.shape}")

print(f"Human Posts (HF): {df_human_posts.shape}")

AI Posts: (290251, 13)
Human Posts (HF): (272838, 66)


In [10]:
df_ai_posts["Label"] = 0
df_human_posts["Label"] = 1

In [11]:
df_human_posts.columns

Index(['allow_live_comments', 'archived', 'author', 'author_fullname',
       'banned_by', 'category', 'content_categories', 'contest_mode',
       'created_utc', 'discussion_type', 'distinguished', 'domain', 'edited',
       'gilded', 'hidden', 'hide_score', 'id', 'is_created_from_ads_ui',
       'is_crosspostable', 'is_meta', 'is_original_content',
       'is_reddit_media_domain', 'is_robot_indexable', 'is_self', 'is_video',
       'locked', 'media', 'media_embed', 'media_only', 'name', 'no_follow',
       'num_comments', 'num_crossposts', 'over_18', 'parent_whitelist_status',
       'permalink', 'pinned', 'post_hint', 'pwls', 'quarantine', 'removed_by',
       'removed_by_category', 'retrieved_on', 'score', 'secure_media',
       'secure_media_embed', 'selftext', 'send_replies', 'spoiler', 'stickied',
       'subreddit_id', 'subreddit_name_prefixed', 'subreddit_subscribers',
       'subreddit_type', 'suggested_sort', 'title', 'top_awarded_type',
       'total_awards_received', 'trea

In [12]:
df_human_posts["full_post"] = df_human_posts["title"].fillna("") + " " + df_human_posts["selftext"].fillna("")

In [13]:
df_ai_posts.columns

Index(['id', 'title', 'content', 'url', 'upvotes', 'downvotes',
       'comment_count', 'created_at', 'submolt_id', 'submolt_name',
       'submolt_display_name', 'author_id', 'author_name', 'Label'],
      dtype='object')

In [14]:
df_ai_posts["full_post"] = df_ai_posts["title"].fillna("") + " " + df_ai_posts["content"].fillna("")

In [ ]:

df_ai = df_ai_posts[["full_post", "Label"]]



In [16]:
df_human = df_human_posts[["full_post", "Label"]]

In [17]:
final_df = pd.concat([df_ai, df_human], ignore_index=True)

In [19]:
final_df.to_csv(f"{DATA_DIR}/final_dataset_uncleaned.csv", index=False)